# WHERE vs HAVING

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-02-where-vs-having.ipynb)

`WHERE` and `HAVING` both filter, but they run at different stages of a query. `WHERE` filters **rows** before
anything is grouped. `HAVING` filters **groups** after the aggregates are computed. Putting a condition in the
wrong place either fails or answers a different question. This chapter explains the difference with examples and
ends with practice questions and answers.

The basics chapter introduced both keywords briefly. Start there if `GROUP BY` is new to you.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

## 1. The two stages

The engine processes a grouped query in this order:

1. `WHERE` looks at one row at a time. It cannot see an aggregate such as `SUM(amount)`, because the sum does not
   exist yet.
2. `GROUP BY` forms the groups, and the aggregates are computed.
3. `HAVING` looks at one group at a time. It can use aggregates and grouped columns.

A practical test: if the condition is about a single order, it goes in `WHERE`. If it is about a customer's
total or count, it goes in `HAVING`.

## 2. A condition on rows: WHERE

Completed orders only. The condition is about each order, so it goes in `WHERE`.

In [ ]:
spark.sql("""
SELECT customer_id, COUNT(*) AS completed_orders
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY customer_id
""").show()

## 3. A condition on groups: HAVING

Customers whose completed total is above 500. The filter is about the total, so it goes in `HAVING`. The
`status` condition is still about each order, so it stays in `WHERE`.

In [ ]:
spark.sql("""
SELECT customer_id, SUM(amount) AS completed_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING SUM(amount) > 500
ORDER BY customer_id
""").show()

## 4. Filtering before grouping changes the answer

The rows you keep in `WHERE` are the rows every aggregate sees. Compare "customers with at least two orders" with
and without the completed-only filter. Customers 4 and 6 each have only one completed order, so they drop out.

In [ ]:
spark.sql("""
SELECT customer_id, COUNT(*) AS orders
FROM orders
GROUP BY customer_id
HAVING COUNT(*) >= 2
ORDER BY customer_id
""").show()

spark.sql("""
SELECT customer_id, COUNT(*) AS completed_orders
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING COUNT(*) >= 2
ORDER BY customer_id
""").show()

## 5. Common mistakes

**An aggregate in WHERE.** This query is rejected, because `SUM(amount)` does not exist yet at the `WHERE` stage:

```sql
SELECT customer_id FROM orders WHERE SUM(amount) > 500 GROUP BY customer_id;  -- error
```

Move the condition to `HAVING`.

**A row condition in HAVING.** `HAVING status = 'completed'` fails, because `status` is neither grouped nor
aggregated. Put row conditions in `WHERE`.

**HAVING without GROUP BY.** The whole table is one group, so `HAVING` tests that single total. It is a handy
guard: fail the query when a total is suspiciously small.

In [ ]:
spark.sql("""
SELECT SUM(amount) AS total_amount
FROM orders
HAVING SUM(amount) > 1000
""").show()

## Summary

| Question | Keyword |
| --- | --- |
| Is this one order completed? | `WHERE` |
| Does this customer's total exceed 500? | `HAVING` |
| Does this customer have more than one order? | `HAVING COUNT(*) > 1` |
| Did this customer place any order at all? | neither: use a join or `EXISTS` |

## Practice questions

1. Which customers have a total order amount above 500, counting every status? Show `customer_id` and the total.
2. Which customers have more than one order that is not cancelled? Show `customer_id` and the count.
3. For each status, what is the average order amount? Show only the statuses whose average is above 250. Round to 2 decimals.
4. Why does `WHERE SUM(amount) > 500` fail? Explain in one sentence.

## Answers

### Question 1

In [ ]:
q = """
SELECT customer_id, SUM(amount) AS total
FROM orders
GROUP BY customer_id
HAVING SUM(amount) > 500
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 950), (4, 540), (5, 950), (6, 600)]

### Question 2

In [ ]:
q = """
SELECT customer_id, COUNT(*) AS orders
FROM orders
WHERE status <> 'cancelled'
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 3), (2, 2), (3, 2), (4, 2), (5, 2)]

### Question 3

In [ ]:
q = """
SELECT status, ROUND(AVG(amount), 2) AS average_amount
FROM orders
GROUP BY status
HAVING AVG(amount) > 250
ORDER BY status
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 287.5), ("completed", 272.27)]

### Question 4

`WHERE` runs before grouping, so it cannot use an aggregate. `SUM(amount)` only exists after `GROUP BY`, which is
the stage `HAVING` belongs to.